# Mechanism-of-action prediction from L1000 transcriptomics
### cureATime / DFKI omics challenge — analysis notebook

Predict which of 15 mechanism classes a drug hits, from 978 landmark genes measured in
treated A549 lung cancer cells.

This notebook walks the analysis from raw data to the final submission, including the
two findings that mattered: **RAF inhibitors paradoxically activate the pathway they
block**, and **our decision rule was optimising the wrong metric**.

Final cross-validated **macro-F1 0.728 ± 0.014**, from a 0.629 baseline.

## 0. Setup

Point `DATA` at the directory holding `X_train.csv.gz`. `N_SEEDS` controls how many repeated CV splits to run — 3 is quick (~1 min), 10 reproduces the reported numbers (~4 min).

In [ ]:
import warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score
from sklearn.metrics import precision_recall_fscore_support
warnings.filterwarnings("ignore")

DATA    = "../omics"      # <-- edit me
N_SEEDS = 3
CTRL    = "BRD-K60230970"
HI_DOSES, LOW_DOSES, TOP_DOSE = [3.33, 10.0], [0.04, 0.12, 0.37, 1.11], 10.0

## 1. Load the data, and audit it first

The first thing to check is replication. One compound dominates the training set.

In [ ]:
X_tr = pd.read_csv(f"{DATA}/X_train.csv.gz"); X_te = pd.read_csv(f"{DATA}/X_test.csv.gz")
X_un = pd.read_csv(f"{DATA}/X_unlabeled.csv.gz")
y_tr = pd.read_csv(f"{DATA}/y_train.csv");  m_tr = pd.read_csv(f"{DATA}/meta_train.csv")
m_te = pd.read_csv(f"{DATA}/meta_test.csv")
CLASSES = [l.strip() for l in open(f"{DATA}/classes.txt") if l.strip()]
GENES   = [c for c in X_tr.columns if c != "sample_id"]
ci = {c: i for i, c in enumerate(CLASSES)}; K_ = len(CLASSES)

meta_all = m_tr.merge(y_tr, on="sample_id")
vc = meta_all.compound.value_counts()
print(f"{meta_all.compound.nunique()} compounds, {len(meta_all)} signatures, {len(GENES)} genes")
print(f"\nmost replicated compound: {vc.index[0]}  ->  {vc.iloc[0]} signatures")
sub = meta_all[meta_all.compound == CTRL]
print(f"  class     : {sub.moa.unique()[0]}")
print(f"  doses     : {sub.dose_um.unique()}")
print(f"  plates    : {sub.sample_id.str.split(':').str[0].nunique()}")
print(f"  cv_fold   : {sub.cv_fold.unique()}")

`BRD-K60230970` is a **replicated plate control**: 300 signatures, all at 20 µM, spread over
25 plates, all assigned to CV fold 2. It is labelled proteasome inhibitor.

Left in, 94% of the proteasome class is one molecule, and three of the four CV folds see its
300 rows while the fourth does not. We drop it. Note what that leaves.

In [ ]:
meta = meta_all[meta_all.compound != CTRL].reset_index(drop=True)
print(f"after dropping the control: {meta.compound.nunique()} compounds, {len(meta)} signatures\n")
print(meta.groupby("moa").compound.nunique().sort_values().to_string())

Four to eleven compounds per class, eight classes with exactly four, and **proteasome down to
three**. Because macro-F1 weights all 15 classes equally, the rare classes drive the score —
and with grouped 4-fold CV, a four-compound class puts exactly one compound in each fold, so a
single flipped prediction moves macro-F1 by ~1/15.

## 2. Calibration and compound profiles

Each gene is recentred and rescaled by its median and MAD across the 1,279 unlabelled compounds,
which serve as an "inert" reference. A compound profile is the mean of its 3.33 and 10 µM
signatures.

In [ ]:
def compound_profiles(X, md):
    df = md[["sample_id", "compound", "dose_um"]].merge(X, on="sample_id")
    return df[df.dose_um.isin(HI_DOSES)].groupby("compound")[GENES].mean()

un   = X_un[X_un.compound != CTRL]
prof = compound_profiles(un[["sample_id"] + GENES], un[["sample_id", "compound", "dose_um"]])
MED  = prof.median(0).values
MAD  = (prof - prof.median(0)).abs().median(0).values * 1.4826
MAD  = np.where(MAD < 1e-6, 1.0, MAD)

P_tr, P_te = compound_profiles(X_tr, meta), compound_profiles(X_te, m_te)
Z_tr, Z_te = (P_tr.values - MED) / MAD, (P_te.values - MED) / MAD
comps, test_comps = P_tr.index.to_numpy(), P_te.index.to_numpy()
y = meta.groupby("compound").moa.first().reindex(comps).to_numpy()

sig    = meta[["sample_id", "compound", "dose_um", "moa"]].merge(X_tr, on="sample_id")
Z_sig  = (sig[GENES].values - MED) / MAD
te_sig = m_te[["sample_id", "compound", "dose_um"]].merge(X_te, on="sample_id")
Z_te_sig = (te_sig[GENES].values - MED) / MAD
print("compound profiles:", Z_tr.shape, "| signatures:", Z_sig.shape, "| test:", Z_te.shape)

## 3. The biology: why RAF looked unpredictable

RAF was the worst class in every model — zero reproducible marker genes, negative within-class
coherence, no same-class nearest neighbour, recall 0.00.

The standard expectation is that a RAF inhibitor *suppresses* MAPK output. We build an **ERK
output score** from the six landmark genes MEK inhibition suppresses hardest, then look at it
**dose by dose** instead of averaging.

In [ ]:
ERK = {"DUSP6": "208891_at", "DUSP4": "204014_at", "IER3": "201631_s_at",
       "ETV1": "206501_x_at", "FOSL1": "204420_at", "SPRED2": "212458_at"}
gi = [GENES.index(p) for p in ERK.values()]

def dose_curves(Z, comp, dose):
    d = pd.DataFrame({"compound": comp, "dose": dose, "s": Z[:, gi].mean(1)})
    piv = d.pivot_table(index="compound", columns="dose", values="s")
    return piv.T.interpolate(limit_direction="both").T   # one compound has only 5 doses

curves = dose_curves(Z_sig, sig.compound.values, sig.dose_um.values).reindex(comps)
cls = pd.Series(y, index=comps)
show = ["RAF inhibitor", "MEK inhibitor", "EGFR inhibitor"]
print("ERK output score by dose (class medians)\n")
print(curves.groupby(cls).median().loc[show].round(2).to_string())

**MEK inhibitors suppress from the lowest dose. RAF inhibitors do the opposite, then reverse.**

Per RAF compound:

In [ ]:
print(curves[cls == "RAF inhibitor"].round(2).to_string())

Three of four rise above control at low/intermediate dose and cross to negative at 10 µM. The
fourth (`BRD-K56343971`) never crosses — a pharmacologically distinct RAF inhibitor.

This is **paradoxical RAF activation**: in a KRAS-mutant, BRAF-wild-type line, partial RAF
occupancy promotes dimerisation and *raises* ERK output; only at high concentration does
inhibition win.

Now quantify it. The index needs both a positive low-dose peak **and** a fall at top dose — that
excludes MEK and HSP, which are negative throughout.

In [ ]:
def paradox_index(cv):
    mx = cv[LOW_DOSES].max(1)
    return np.minimum(mx, mx - cv[TOP_DOSE])

paradox = paradox_index(curves)
is_raf  = (y == "RAF inhibitor").astype(int)
erk_avg = pd.DataFrame(Z_tr, index=comps, columns=GENES)[list(ERK.values())].mean(1)

print(f"AUC, RAF vs all other classes")
print(f"  dose-resolved crossover index      {roc_auc_score(is_raf, paradox):.3f}")
print(f"  ERK score, all 6 doses averaged    {roc_auc_score(is_raf, curves.mean(1)):.3f}")
print(f"  ERK score on the 3.33+10 profile   {roc_auc_score(is_raf, erk_avg):.3f}   <-- chance")

rng = np.random.default_rng(0); v = paradox.values
null = np.array([v[rng.permutation(len(v))[:4]].mean() for _ in range(20000)])
print(f"\npermutation p = {(null >= v[y=='RAF inhibitor'].mean()).mean():.4f}")

**The standard profile recipe averages 3.33 and 10 µM — directly across the crossover — and
cancels the signal entirely (AUC 0.485, chance).** The dose-resolved index reaches 0.826.

That is why the centroid scored 0.00 on RAF while a signature-level tree model, which sees each
dose as its own row, reached 0.60.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
col = {"RAF inhibitor": "#C2352B", "MEK inhibitor": "#2F5C8A", "EGFR inhibitor": "#9AA5B1"}
dx = sorted(curves.columns)
for c, k in col.items():
    s = curves[cls == c]
    for _, r in s.iterrows(): ax.plot(dx, r[dx], color=k, lw=.9, alpha=.45)
    ax.plot(dx, s[dx].mean(0), color=k, lw=2.4, marker="o", ms=4, label=c.replace(" inhibitor", ""))
ax.axhline(0, color="grey", ls="--", lw=.8); ax.axvspan(3.33, 10, color="#C2352B", alpha=.07)
ax.set_xscale("log"); ax.set_xticks(dx); ax.set_xticklabels([str(d) for d in dx])
ax.set_xlabel("dose (µM)"); ax.set_ylabel("ERK output score (mean z)")
ax.set_title("Shaded band = what the profile recipe averages over")
ax.legend(frameon=False); plt.show()

### Does this generalise? No — and the failure is the useful part

Same recipe, p53 footprint, for PARP.

In [ ]:
P53 = ["202284_s_at", "203409_at", "204781_s_at", "218346_s_at", "208478_s_at", "203725_at"]
pi  = [GENES.index(p) for p in P53]
pc  = pd.DataFrame({"compound": sig.compound.values, "dose": sig.dose_um.values,
                    "s": Z_sig[:, pi].mean(1)}).pivot_table(index="compound", columns="dose", values="s")
pc  = pc.T.interpolate(limit_direction="both").T.reindex(comps)
is_parp = (y == "PARP inhibitor").astype(int)
print(pc.groupby(cls).median().loc[["PARP inhibitor", "topoisomerase inhibitor"]].round(2).to_string())
print(f"\nAUC PARP vs rest")
print(f"  dose-averaged p53 score   {roc_auc_score(is_parp, pd.DataFrame(Z_tr,index=comps,columns=GENES)[P53].mean(1)):.3f}")
print(f"  dose-resolved peak        {roc_auc_score(is_parp, -pc.max(1)):.3f}")

PARP's p53 response is **monotonic and weak**, emerging only at the top dose, while topoisomerase
sits high from the lowest. Dose-resolving it *loses* information; averaging keeps it.

> **Principle.** Dose resolution recovers signal when the response changes sign, and destroys it
> when the response is weak but monotonic. Averaging is a noise reducer — it only costs you
> something when it averages across a crossover.

## 4. The two models

They fail on different classes, which is why we blend them.

**Arm 1 — cosine centroid** with *soft pathway-superclass labels*: instead of a hard one-hot
target, 15% of each label's mass goes to its pathway siblings. A class with four noisy compounds
gives a poor direction estimate; borrowing from siblings regularises it. Plus fold-internal
logistic evidence terms for PARP (p53 + amplitude) and RAF (the paradox index).

In [ ]:
SUPER = {"PI3K inhibitor":"PI3Kaxis","AKT inhibitor":"PI3Kaxis","mTOR inhibitor":"PI3Kaxis",
         "EGFR inhibitor":"MAPK","RAF inhibitor":"MAPK","MEK inhibitor":"MAPK",
         "topoisomerase inhibitor":"DNAdamage","PARP inhibitor":"DNAdamage",
         "tubulin polymerization inhibitor":"mitotic","PLK inhibitor":"mitotic",
         "CDK inhibitor":"cellcycle","HDAC inhibitor":"chromatin",
         "proteasome inhibitor":"proteostasis","HSP inhibitor":"proteostasis",
         "JAK inhibitor":"cytokine"}

def soft_targets(lab, eps=0.15):
    T = np.zeros((len(lab), K_))
    for i, c in enumerate(lab):
        sibs = [d for d in CLASSES if SUPER[d] == SUPER[c] and d != c]
        T[i, ci[c]] = 1 - eps if sibs else 1.0
        for s in sibs: T[i, ci[s]] = eps / len(sibs)
    return T

def unit(Z):
    n = np.linalg.norm(Z, axis=1, keepdims=True); return Z / np.where(n < 1e-9, 1, n)

def centroid_proba(Ztr, ytr, Zq, eps=0.15, temp=0.1):
    cen = unit(soft_targets(ytr, eps).T @ unit(Ztr))
    S = unit(Zq) @ cen.T; E = np.exp((S - S.max(1, keepdims=True)) / temp)
    return E / E.sum(1, keepdims=True)

def evidence(Xf, yb, Xq):
    return LogisticRegression(max_iter=1000, class_weight="balanced").fit(Xf, yb).predict_proba(Xq)[:, 1]

amp = lambda Z, c, idx: np.log10(pd.Series((np.abs(Z) > 2).sum(1), index=c).groupby(level=0).max().reindex(idx).values + 1)
F_parp = np.c_[pd.DataFrame(Z_tr, index=comps, columns=GENES)[P53].mean(1).values,
               amp(Z_sig, sig.compound.values, comps)]
F_raf  = paradox.values[:, None]
print("evidence features built:", F_parp.shape, F_raf.shape)

**Arm 2 — LightGBM at signature level.** Trained on all 515 signature rows rather than 86
averaged profiles. Six-fold more training rows is what the trees needed; the winning config
uses heavy column subsampling (`colsample 0.24`), i.e. few genes per tree.

In [ ]:
LGB = dict(n_estimators=456, num_leaves=5, max_depth=3, learning_rate=0.085,
           min_child_samples=5, subsample=0.82, subsample_freq=1, colsample_bytree=0.24,
           reg_lambda=7.69, class_weight="balanced", objective="multiclass",
           num_class=K_, verbose=-1, n_jobs=4)

def make_folds(seed, labels=None, n=4):
    labels = y if labels is None else labels
    rng = np.random.default_rng(seed); f = np.empty(len(labels), int)
    for c in np.unique(labels):
        idx = np.where(labels == c)[0]; rng.shuffle(idx); f[idx] = np.arange(len(idx)) % n
    return f
print("grouped, class-stratified folds: each compound held out once per seed")

## 5. Cross-validation

Compound-grouped 4-fold, repeated over re-randomised splits. Both arms, the blend, and the
decision rule are all evaluated on the same splits so differences are paired.

In [ ]:
def oof(seed):
    f = make_folds(seed); Pg = np.zeros((len(comps), K_)); Pc = np.zeros_like(Pg)
    for k in range(4):
        tr, te = f != k, f == k
        rows = np.isin(sig.compound.values, comps[tr]); q = np.isin(sig.compound.values, comps[te])
        m = lgb.LGBMClassifier(**LGB).fit(Z_sig[rows], np.array([ci[c] for c in sig.moa[rows]]))
        d = pd.DataFrame(m.predict_proba(Z_sig[q])); d["c"] = sig.compound.values[q]
        Pg[te] = d.groupby("c").mean().reindex(comps[te]).values
        P = centroid_proba(Z_tr[tr], y[tr], Z_tr[te]).copy()
        P[:, ci["PARP inhibitor"]] = .9*P[:, ci["PARP inhibitor"]] + .1*evidence(F_parp[tr], (y[tr]=="PARP inhibitor").astype(int), F_parp[te])
        P[:, ci["RAF inhibitor"]]  = .7*P[:, ci["RAF inhibitor"]]  + .3*evidence(F_raf[tr],  (y[tr]=="RAF inhibitor").astype(int),  F_raf[te])
        Pc[te] = P / P.sum(1, keepdims=True)
    return Pg, Pc

SEEDS = list(range(2, 2 + N_SEEDS))
OOF = {s: oof(s) for s in SEEDS}
mf = lambda P: f1_score(y, [CLASSES[i] for i in P.argmax(1)], average="macro")
cent = np.array([mf(OOF[s][1]) for s in SEEDS]); gbm = np.array([mf(OOF[s][0]) for s in SEEDS])
bl   = np.array([mf(.5*OOF[s][0] + .5*OOF[s][1]) for s in SEEDS])
for n, v in [("centroid + evidence terms", cent), ("LightGBM (signature level)", gbm), ("50/50 blend", bl)]:
    print(f"{n:<28} {v.mean():.3f} +/- {v.std():.3f}")

## 6. The decision rule was optimising the wrong metric

Every model above picks its answer with `argmax`. **Argmax is the Bayes rule for accuracy.** We are
scored on **macro-F1**, whose per-class optimum sits where precision equals recall. Look at the gap
between the two, then at which classes are out of balance.

In [ ]:
t, p = [], []
for s in SEEDS:
    t += list(y); p += [CLASSES[i] for i in (.5*OOF[s][0] + .5*OOF[s][1]).argmax(1)]
pr, rc, f1, _ = precision_recall_fscore_support(t, p, labels=CLASSES, zero_division=0)
tab = pd.DataFrame({"precision": pr, "recall": rc, "F1": f1},
                   index=[c.replace(" inhibitor", "") for c in CLASSES])
tab["prec/rec"] = (tab.precision / tab.recall.replace(0, np.nan)).round(2)
print(f"accuracy {np.mean(np.array(t)==np.array(p)):.3f}   macro-F1 {f1.mean():.3f}\n")
print(tab.round(3).sort_values("prec/rec", ascending=False).to_string())

PARP, PI3K and JAK are **under-predicted roughly two-fold** — when the model names them it is right
~35% of the time, it just rarely names them. Under F1 that is strictly suboptimal.

The fix is one global parameter, and no change to the model: divide each class score by its training
frequency raised to a power. The exponent is chosen **inside each training fold**, never on the
compounds being scored.

In [ ]:
prior = np.array([(y == c).mean() for c in CLASSES])
rule  = lambda P, a: (P / prior**a).argmax(1)
ALPHAS = [0, .25, .5, .75, 1.0, 1.25]

nested = []
for s in SEEDS:
    f = make_folds(s); P = .5*OOF[s][0] + .5*OOF[s][1]; out = np.empty(len(comps), object)
    for k in range(4):
        tr, te = np.where(f != k)[0], np.where(f == k)[0]
        best = max(ALPHAS, key=lambda a: f1_score(y[tr], [CLASSES[i] for i in rule(P[tr], a)], average="macro"))
        out[te] = [CLASSES[i] for i in rule(P[te], best)]
    nested.append(f1_score(y, list(out), average="macro"))
nested = np.array(nested)
print(f"argmax blend            {bl.mean():.3f} +/- {bl.std():.3f}")
print(f"prior-corrected (nested){nested.mean():>8.3f} +/- {nested.std():.3f}")
print(f"delta {(nested-bl).mean():+.3f}, wins {(nested>bl).sum()}/{len(SEEDS)}")
print("\n(reported run, N_SEEDS=10: 0.728 +/- 0.014, +0.029, 10/10;")
print(" 0.715 on ten further seeds held out from selection)")

> **Caution.** Fitting a *separate* weight per class looks even better — 0.750 under
> leave-one-seed-out — but every seed reuses the same 86 compounds, so the weights memorise which
> compounds are hard. Refitted inside each compound fold it collapses to 0.679, worse than the flat
> blend. **Validate by compound, not by seed.**

## 7. Final submission

Fit both arms on all 86 compounds, blend, apply the decision rule.

In [ ]:
ALPHA = 0.5
erk_te_c = dose_curves(Z_te_sig, te_sig.compound.values, te_sig.dose_um.values).reindex(test_comps)
F_raf_te  = paradox_index(erk_te_c).values[:, None]
F_parp_te = np.c_[pd.DataFrame(Z_te, index=test_comps, columns=GENES)[P53].mean(1).values,
                  amp(Z_te_sig, te_sig.compound.values, test_comps)]

Pc = centroid_proba(Z_tr, y, Z_te).copy()
Pc[:, ci["PARP inhibitor"]] = .9*Pc[:, ci["PARP inhibitor"]] + .1*evidence(F_parp, (y=="PARP inhibitor").astype(int), F_parp_te)
Pc[:, ci["RAF inhibitor"]]  = .7*Pc[:, ci["RAF inhibitor"]]  + .3*evidence(F_raf,  (y=="RAF inhibitor").astype(int),  F_raf_te)
Pc /= Pc.sum(1, keepdims=True)

m = lgb.LGBMClassifier(**LGB).fit(Z_sig, np.array([ci[c] for c in sig.moa]))
d = pd.DataFrame(m.predict_proba(Z_te_sig)); d["c"] = te_sig.compound.values
Pg = d.groupby("c").mean().reindex(test_comps).values

P = (.5*Pg + .5*Pc) / prior**ALPHA; P /= P.sum(1, keepdims=True)
out = pd.DataFrame(P, columns=CLASSES); out.insert(0, "compound", test_comps)
out.to_csv("submission_final_v2.csv", index=False)

assert list(out.columns) == ["compound"] + CLASSES and np.allclose(out[CLASSES].sum(1), 1.0)
print(f"wrote submission_final_v2.csv — {len(out)} rows, rows sum to 1.0\n")
print(pd.Series([CLASSES[i] for i in P.argmax(1)]).value_counts().to_string())

## Summary

| | macro-F1 |
|---|---|
| cosine centroid, hard labels | 0.629 |
| + soft pathway-superclass labels | 0.672 |
| + PARP and RAF evidence terms | 0.689 |
| LightGBM at signature level | 0.688 |
| 50/50 blend | 0.699 |
| **+ prior-corrected decision rule** | **0.728** |

Bootstrap 95% CI ≈ **[0.61, 0.78]** — wider than any difference between the last three rows. With
29 test compounds (~2 per class) a single compound moves macro-F1 by several points, so treat the
leaderboard number as noisy.

Four classes stay poorly predicted — PARP, JAK, PI3K, RAF. That is because their compounds do not
resemble each other transcriptionally, not because the models are inadequate. JAK in particular is
a pharmacological limit: A549 without cytokine stimulation has little baseline JAK-STAT tone, so a
JAK inhibitor has nothing to switch off.